[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap09/cap09.EPs_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)


## 💻 **Parte Prática com Exercícios de Programação**

🚧 **Em construção!**

A presente lista de exercícios de programação (EP) consolida as formulações teóricas apresentadas ao longo do Capítulo 9 — *Deep Learning* para Visão Computacional — por meio de uma trilha prática aplicada. Diferentemente do treinamento de redes neurais completas com PyTorch, que exige tempo de execução e, por vezes, GPU, os EPs deste capítulo isolam as **grandezas intermediárias** de um *pipeline* real de aprendizado profundo — a saída de uma única camada convolucional, o resultado de uma operação de *pooling*, a contagem de parâmetros treináveis de uma arquitetura, a sobreposição entre caixas delimitadoras candidatas e o filtro de supressão de não-máximos — permitindo validar manualmente cada etapa do raciocínio sem depender de bibliotecas de aprendizado de máquina nem de treinamento real.

O encadeamento dos exercícios reproduz o fluxo conceitual do capítulo: inicia-se com o cálculo manual da saída de uma **camada convolucional aprendida**, a partir de um *kernel* e um viés já treinados; avança-se para a operação de ***pooling*** (máximo e média), que reduz a resolução espacial entre blocos convolucionais; prossegue-se com a **contagem de parâmetros treináveis** de uma arquitetura CNN completa, evidenciando por que o compartilhamento de pesos torna essas redes tão mais econômicas que uma camada totalmente conectada equivalente; aprofunda-se no cálculo de **Interseção sobre União (IoU)** e na **Supressão de Não-Máximos (NMS)**, etapa de pós-processamento comum a detectores como o Faster R-CNN e o YOLO; e conclui-se com um ***pipeline* integrado**, unindo a saída de um detector de objetos (após NMS) a uma medição do mundo real por referência de escala — o mesmo princípio da fotogrametria estudada na integração final do capítulo.

> ### ❗ Diretrizes para a Resolução dos Exercícios de Programação
>
> Em todos os exercícios deste capítulo, as etapas de discretização ou arredondamento numérico devem empregar o arredondamento padrão para o inteiro mais próximo (*round half away from zero*), mitigando ambiguidades em valores com fração exatamente igual a $0{,}5$. Salvo indicação explícita em contrário: (i) a operação de "convolução" segue a convenção adotada pelos *frameworks* de aprendizado profundo — **correlação cruzada**, sem inversão espacial do *kernel*, exatamente como apresentado na Seção "Convolução como Camada Aprendida"; (ii) o preenchimento (*padding*) é feito com zeros; (iii) caixas delimitadoras são especificadas no formato canto-a-canto $(x_1, y_1, x_2, y_2)$, com $x_1 < x_2$ e $y_1 < y_2$; e (iv) vetores/matrizes seguem indexação a partir de $0$, com a convenção `[linha][coluna]` para estruturas bidimensionais.


### 🎯 Objetivo deste Caderno

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.


#### Download

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:


In [1]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")


✅ Ambiente pronto. Morph: 1.1.7 | TestSuite: 1.1.2


#### Executando os Testes
Para avaliar os testes, execute `TestSuite("EP09_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

Para testar código Python diretamente, sem salvar arquivo, use `run_code(codigo)` passando o código como *string* numa variável `codigo`:

```python
codigo = """
# ... seu código aqui ...
"""
TestSuite("EP09_01").run_code(codigo)
```


### EP09_01 🟢 Convolução 2D Manual (Forward de uma Camada Aprendida)

O PyTorch, apresentado neste capítulo, executa `nn.Conv2d(x)` em uma única chamada — mas por trás dessa chamada está apenas a operação de correlação cruzada entre um *kernel* (já treinado) e uma vizinhança da entrada, seguida da soma de um viés e de uma ativação, exatamente como formalizado na Seção "Convolução como Camada Aprendida". A diferença essencial em relação à convolução fixa do Capítulo 3 é que, aqui, os valores do *kernel* e do viés **já vêm prontos** (como se tivessem sido aprendidos por gradiente), e cabe a você reproduzir manualmente a passagem direta (*forward pass*) que o *framework* executa internamente.

Antes de treinar uma CNN de verdade, você foi encarregado de implementar essa passagem direta do zero, para uma única camada convolucional com um único canal de entrada e um único filtro de saída, incluindo suporte a *padding* e *stride* arbitrários.

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler as dimensões $H \times W$ do mapa de características de entrada e, em seguida, seus $H \times W$ valores reais.
2. **Kernel e viés:** Ler as dimensões $k_h \times k_w$ do *kernel* (já treinado), seus valores reais, e o viés $b$ (real, escalar).
3. **Hiperparâmetros:** Ler o *padding* $p$ (inteiro, número de zeros adicionados em cada borda) e o *stride* $s$ (inteiro, passo do deslizamento).
4. **Preenchimento:** Adicionar $p$ zeros em cada uma das quatro bordas do mapa de entrada antes da correlação.
5. **Correlação cruzada:** Para cada posição de saída $(i, j)$, calcular
$$
z(i,j) = b + \sum_{u=0}^{k_h-1} \sum_{v=0}^{k_w-1} K(u,v) \cdot X_{pad}(i \cdot s + u,\; j \cdot s + v),
$$
   percorrendo a entrada **sem** inverter o *kernel* (convenção dos *frameworks* de aprendizado profundo, diferente da convolução matemática clássica).
6. **Ativação:** Aplicar ReLU a cada valor: $a(i,j) = \max(0, z(i,j))$.
7. **Dimensões de saída:** $O_h = \lfloor (H + 2p - k_h)/s \rfloor + 1$ e $O_w = \lfloor (W + 2p - k_w)/s \rfloor + 1$.
8. **Saída:** Imprimir $O_h$ e $O_w$ na primeira linha, seguidos de $O_h$ linhas com $O_w$ valores reais cada (o mapa de características de saída, já com ReLU aplicada), formatados com 4 casas decimais.

#### 📌 Restrições Computacionais

* **Um canal de entrada, um filtro de saída:** não é necessário lidar com múltiplos canais ou múltiplos filtros nesta versão simplificada.
* **Sem inversão do *kernel*:** implemente correlação cruzada, não a convolução matemática clássica com *kernel* invertido — é essa a operação que o PyTorch (e a maioria dos *frameworks*) chama de "convolução".
* **Preenchimento por zeros:** os $p$ pixels adicionados em cada borda valem sempre $0$.
* **Formatação:** todos os valores de saída devem ter exatamente 4 casas decimais, mesmo quando o valor é inteiro (ex.: `2.0000`).

#### 🧠 Fundamentação Teórica

| Elemento | Papel na camada convolucional |
|---|---|
| *Kernel* $K$ | Parâmetros aprendidos por gradiente, análogos aos filtros de Sobel do Capítulo 3, mas ajustados a partir de dados |
| Viés $b$ | Parâmetro aprendido, somado à saída de cada janela |
| *Padding* $p$ | Controla a dimensão espacial da saída; $p=0$ ("*valid*") reduz a resolução, $p$ maior preserva-a |
| *Stride* $s$ | Passo do deslizamento; $s>1$ sub-amostra a saída, reduzindo custo computacional |
| ReLU | Ativação não-linear aplicada após cada convolução, sem a qual camadas empilhadas colapsariam em uma única transformação linear |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiros $H$ e $W$.
* Próximas $H$ linhas: $W$ valores reais cada (mapa de entrada).
* Próxima linha: Inteiros $k_h$ e $k_w$.
* Próximas $k_h$ linhas: $k_w$ valores reais cada (*kernel*).
* Próxima linha: Um valor real $b$ (viés).
* Próxima linha: Inteiros $p$ e $s$ (*padding* e *stride*).

**Saída:**

* Linha 1: Inteiros $O_h$ e $O_w$.
* Próximas $O_h$ linhas: $O_w$ valores reais cada, formatados com 4 casas decimais.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 1<br>1 1<br>-2<br>0 1 | 2 2<br>2.0000 3.0000<br>0.0000 2.0000 | *Padding* 0, *stride* 1: saída $2\times2$ sem preenchimento. |
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 0<br>0 1<br>0<br>1 2 | 2 2<br>1.0000 0.0000<br>1.0000 2.0000 | *Padding* 1, *stride* 2: entrada preenchida com zeros antes da correlação. |


In [2]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0901" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Convolução 2D Manual</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 correlação cruzada + viés + ReLU</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Entrada 4×4 fixa, kernel 2×2 fixo (destacado em azul), padding=0, stride=1 &mdash; deslize para escolher a posição de saída.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Posição de saída (i,j)</label>
        <span id="ep0901_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
      </div>
      <input id="ep0901_sl" style="width:100%;accent-color:#2980b9;" max="8" min="0" step="1" type="range" value="0">
    </div>
    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Entrada X (4×4)</div>
        <div id="ep0901_grid" style="display:grid;grid-template-columns:repeat(4,44px);gap:3px;"></div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Kernel K (2×2)</div>
        <div id="ep0901_kernel" style="display:grid;grid-template-columns:repeat(2,44px);gap:3px;"></div>
      </div>
    </div>
    <div id="ep0901_debug" style="margin-top:20px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var X = [[1,3,2,0],[0,1,4,1],[2,0,1,3],[1,2,0,1]];
    var K = [[1,0],[0,-1]];
    var bias = 0.5;
    var outSize = 3; // (4-2)/1+1

    var slEl = root.querySelector('#ep0901_sl');
    var vlEl = root.querySelector('#ep0901_vl');
    var gridEl = root.querySelector('#ep0901_grid');
    var kernelEl = root.querySelector('#ep0901_kernel');
    var dbg = root.querySelector('#ep0901_debug');

    kernelEl.innerHTML = '';
    for(var u=0;u<2;u++) for(var v=0;v<2;v++){
      var kd = document.createElement('div');
      kd.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;background:#bbdefb;border:1px solid #64b5f6;border-radius:6px;font-family:monospace;font-weight:bold;color:#0d47a1;';
      kd.textContent = K[u][v];
      kernelEl.appendChild(kd);
    }

    function render(){
      var pos = parseInt(slEl.value);
      var i = Math.floor(pos/outSize), j = pos%outSize;
      vlEl.textContent = '('+i+','+j+')';
      gridEl.innerHTML = '';
      var soma = 0;
      for(var r=0;r<4;r++){
        for(var c=0;c<4;c++){
          var dentroJanela = (r>=i && r<i+2 && c>=j && c<j+2);
          var d = document.createElement('div');
          d.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:13px;' +
            (dentroJanela ? 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;' : 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;');
          d.textContent = X[r][c];
          gridEl.appendChild(d);
          if(dentroJanela) soma += X[r][c]*K[r-i][c-j];
        }
      }
      var z = soma + bias;
      var a = Math.max(0, z);
      dbg.textContent = 'janela em ('+i+','+j+')  |  soma(X⊙K)='+soma.toFixed(2)+'  +  viés='+bias.toFixed(2)+'  =  z='+z.toFixed(2)+'  →  ReLU(z)='+a.toFixed(4);
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0901');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.1:** Simulador: Convolução 2D Manual (correlação cruzada + viés + ReLU)


In [3]:
%%writefile EP09_01.py
# Código Python


Writing EP09_01.py


In [4]:
TestSuite("EP09_01.py").run()


### EP09_02 🟢 *Pooling* Manual (Máximo e Média)

Entre blocos convolucionais, a arquitetura típica de uma CNN intercala camadas de ***pooling***, que reduzem a resolução espacial do mapa de características sem introduzir novos parâmetros treináveis — ao contrário da convolução, o *pooling* não tem pesos: ele apenas resume cada janela da entrada a um único valor, por um máximo ou por uma média.

Você foi encarregado de implementar essa operação a partir de uma janela deslizante quadrada, sem sobreposição parcial nas bordas (apenas janelas completas), suportando os dois tipos mais comuns: `max` (preserva o valor mais saliente, tipicamente usado para reter bordas e texturas fortes) e `avg` (suaviza a região, preservando informação de intensidade média).

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler as dimensões $H \times W$ do mapa de características de entrada e seus $H \times W$ valores reais.
2. **Janela:** Ler os inteiros $k$ (tamanho da janela quadrada $k \times k$) e $s$ (*stride*).
3. **Tipo:** Ler uma *string*, `max` ou `avg`, indicando o tipo de *pooling*.
4. **Sem preenchimento:** Esta operação **não** utiliza *padding*; janelas que ultrapassariam a borda da entrada são descartadas.
5. **Cálculo:** Para cada posição de saída $(i,j)$, calcular o máximo ou a média dos $k \times k$ valores da janela correspondente, iniciando em $(i \cdot s,\, j \cdot s)$.
6. **Dimensões de saída:** $O_h = \lfloor (H - k)/s \rfloor + 1$ e $O_w = \lfloor (W - k)/s \rfloor + 1$.
7. **Saída:** Imprimir $O_h$ e $O_w$ na primeira linha, seguidos de $O_h$ linhas com $O_w$ valores reais cada, formatados com 4 casas decimais.

#### 📌 Restrições Computacionais

* **Janela quadrada:** $k \times k$, sem suporte a janelas retangulares nesta versão.
* **Sem *padding*:** apenas janelas inteiramente contidas na entrada são consideradas.
* **`avg` usa divisão real:** a média é sempre $\text{soma}/k^2$, mesmo quando o resultado tem muitas casas decimais — arredonde apenas na formatação final, conforme a diretriz geral do capítulo.
* **Formatação:** todos os valores de saída com exatamente 4 casas decimais.

#### 🧠 Fundamentação Teórica

| Elemento | Papel na arquitetura |
|---|---|
| *Pooling* máximo | Preserva a ativação mais forte da janela; comum após camadas convolucionais para reter bordas e texturas salientes |
| *Pooling* médio | Suaviza a região, preservando a intensidade média; comum em camadas finais (*global average pooling*) |
| Ausência de parâmetros | Diferencia o *pooling* da convolução: reduz resolução espacial sem custo adicional de treinamento |
| Redução de resolução | Contribui para a invariância a pequenas translações e para a redução do custo computacional das camadas seguintes |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiros $H$ e $W$.
* Próximas $H$ linhas: $W$ valores reais cada.
* Próxima linha: Inteiros $k$ e $s$.
* Próxima linha: `max` ou `avg`.

**Saída:**

* Linha 1: Inteiros $O_h$ e $O_w$.
* Próximas $O_h$ linhas: $O_w$ valores reais cada, formatados com 4 casas decimais.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>max | 2 2<br>6.0000 4.0000<br>4.0000 5.0000 | *Pooling* máximo, janela $2\times2$, *stride* 2. |
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>avg | 2 2<br>3.7500 2.2500<br>2.2500 2.2500 | *Pooling* médio sobre as mesmas janelas. |


In [5]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0902" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Pooling Manual</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 janela 2×2, stride 2</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Escolha o quadrante e o tipo de pooling para ver o valor resultante.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;display:flex;gap:24px;flex-wrap:wrap;align-items:center;justify-content:center;">
      <div style="flex:1;min-width:180px;">
        <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
          <label style="font-size:12px;font-weight:bold;color:#2980b9;">Quadrante</label>
          <span id="ep0902_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
        </div>
        <input id="ep0902_sl" style="width:100%;accent-color:#2980b9;" max="3" min="0" step="1" type="range" value="0">
      </div>
      <div style="display:flex;gap:8px;">
        <button id="ep0902_max" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #f0ad4e;background:#fff3cd;color:#7a5c00;font-weight:bold;font-size:11px;">max</button>
        <button id="ep0902_avg" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #ddd;background:#f3f4f6;color:#555;font-weight:bold;font-size:11px;">avg</button>
      </div>
    </div>
    <div style="display:flex;justify-content:center;">
      <div id="ep0902_grid" style="display:grid;grid-template-columns:repeat(4,48px);gap:3px;"></div>
    </div>
    <div id="ep0902_debug" style="margin-top:20px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var X = [[1,3,2,4],[5,6,1,2],[2,1,0,3],[4,2,5,1]];
    var tipo = "max";

    var slEl = root.querySelector('#ep0902_sl');
    var vlEl = root.querySelector('#ep0902_vl');
    var gridEl = root.querySelector('#ep0902_grid');
    var dbg = root.querySelector('#ep0902_debug');
    var btnMax = root.querySelector('#ep0902_max');
    var btnAvg = root.querySelector('#ep0902_avg');

    function estiloBotoes(){
      btnMax.style.background = tipo==='max' ? '#fff3cd' : '#f3f4f6';
      btnMax.style.borderColor = tipo==='max' ? '#f0ad4e' : '#ddd';
      btnMax.style.color = tipo==='max' ? '#7a5c00' : '#555';
      btnAvg.style.background = tipo==='avg' ? '#fff3cd' : '#f3f4f6';
      btnAvg.style.borderColor = tipo==='avg' ? '#f0ad4e' : '#ddd';
      btnAvg.style.color = tipo==='avg' ? '#7a5c00' : '#555';
    }

    function render(){
      var q = parseInt(slEl.value);
      var i = Math.floor(q/2)*2, j = (q%2)*2;
      vlEl.textContent = '('+Math.floor(q/2)+','+(q%2)+')';
      gridEl.innerHTML = '';
      var valores = [];
      for(var r=0;r<4;r++){
        for(var c=0;c<4;c++){
          var dentro = (r>=i && r<i+2 && c>=j && c<j+2);
          var d = document.createElement('div');
          d.style.cssText = 'width:48px;height:48px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:13px;' +
            (dentro ? 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;' : 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;');
          d.textContent = X[r][c];
          gridEl.appendChild(d);
          if(dentro) valores.push(X[r][c]);
        }
      }
      var resultado = tipo === 'max' ? Math.max.apply(null, valores) : valores.reduce(function(a,b){return a+b;},0)/valores.length;
      dbg.textContent = 'janela=['+valores.join(', ')+']  |  tipo='+tipo+'  →  resultado='+resultado.toFixed(4);
      estiloBotoes();
    }
    slEl.addEventListener('input', render);
    btnMax.addEventListener('click', function(){ tipo='max'; render(); });
    btnAvg.addEventListener('click', function(){ tipo='avg'; render(); });
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0902');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.2:** Simulador: Pooling Manual (máximo vs. média)


In [6]:
%%writefile EP09_02.py
# Código Python


Writing EP09_02.py


In [7]:
TestSuite("EP09_02.py").run()


### EP09_03 🟡 Contagem de Parâmetros Treináveis de uma CNN

O Exercício 1 deste capítulo pediu para adicionar uma terceira camada convolucional à CNN do Projeto Prático 1 e comparar o número de parâmetros da rede resultante. Este EP formaliza esse cálculo: dada a descrição textual de uma arquitetura — uma sequência de camadas convolucionais, de *pooling* e totalmente conectadas — determine, para cada camada, o número de parâmetros treináveis e o total da rede.

O ponto central deste exercício é notar que a contagem de parâmetros de uma camada convolucional depende **apenas** do tamanho do *kernel* e do número de canais de entrada/saída — **nunca** das dimensões espaciais ($H \times W$) do mapa de características, graças ao **compartilhamento de pesos**: o mesmo *kernel* desliza por toda a imagem, seja ela $28\times28$ ou $280\times280$. É exatamente esse compartilhamento que torna as CNNs muito mais econômicas em parâmetros do que uma camada totalmente conectada equivalente, na qual cada pixel de entrada tem uma conexão (e um peso) independente para cada neurônio de saída.

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler o inteiro $L$ (número de camadas da arquitetura, na ordem em que são aplicadas).
2. **Camadas:** Ler $L$ linhas, cada uma descrevendo uma camada em um dos três formatos:
   * `CONV kh kw cin cout bias` — camada convolucional com *kernel* $k_h \times k_w$, `cin` canais de entrada, `cout` canais (filtros) de saída, e `bias` (0 ou 1) indicando se há viés por filtro;
   * `POOL` — camada de *pooling* (máximo ou médio; não possui parâmetros treináveis);
   * `FC in out bias` — camada totalmente conectada com `in` entradas, `out` saídas, e `bias` (0 ou 1) indicando se há viés por neurônio.
3. **Parâmetros de uma camada `CONV`:** $k_h \cdot k_w \cdot c_{in} \cdot c_{out} + c_{out} \cdot \text{bias}$.
4. **Parâmetros de uma camada `FC`:** $\text{in} \cdot \text{out} + \text{out} \cdot \text{bias}$.
5. **Parâmetros de uma camada `POOL`:** sempre $0$.
6. **Saída:** Para cada camada, na ordem de leitura, imprimir `Camada i: P`, onde $i$ começa em $1$ e $P$ é o número de parâmetros daquela camada. Ao final, imprimir `Total: T`, com $T$ igual à soma de parâmetros de todas as camadas.

#### 📌 Restrições Computacionais

* **Independência da dimensão espacial:** a entrada **não** informa $H \times W$ — a contagem de uma camada `CONV` não depende disso, apenas de `kh kw cin cout`.
* **`bias` sempre 0 ou 1:** multiplique diretamente o termo de viés por esse valor, sem tratamento condicional especial.
* **Camadas `POOL` sem argumentos adicionais:** a linha contém apenas a palavra `POOL`.
* Todos os valores de entrada e saída são inteiros não-negativos.

#### 🧠 Fundamentação Teórica

| Elemento | Papel na contagem de parâmetros |
|---|---|
| Compartilhamento de pesos | O mesmo *kernel* $k_h \times k_w$ é reutilizado em toda a extensão espacial da entrada — por isso a contagem independe de $H \times W$ |
| Canais de entrada/saída | Cada um dos `cout` filtros possui um conjunto de pesos por canal de entrada, daí o fator $c_{in} \cdot c_{out}$ |
| Viés | Um único escalar por filtro (`CONV`) ou por neurônio (`FC`), independente do tamanho do *kernel* ou da entrada |
| Camada `FC` | Cada uma das `in` entradas conecta-se a cada uma das `out` saídas — sem compartilhamento, o que explica seu custo tipicamente muito maior em número de parâmetros |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Próximas $L$ linhas: descrição de cada camada, no formato `CONV kh kw cin cout bias`, `POOL`, ou `FC in out bias`.

**Saída:**

* $L$ linhas no formato `Camada i: P`.
* Última linha: `Total: T`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>CONV 3 3 1 8 1<br>POOL<br>FC 1352 10 1 | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 13530<br>Total: 13610 | Rede com uma convolução, um pooling e uma camada final. |
| 4<br>CONV 3 3 1 8 1<br>CONV 3 3 8 16 1<br>POOL<br>FC 64 32 0 | Camada 1: 80<br>Camada 2: 1168<br>Camada 3: 0<br>Camada 4: 2048<br>Total: 3296 | Rede com duas convoluções empilhadas, como no Projeto Prático 1; a última camada não usa viés. |


In [8]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0903" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Contagem de Parâmetros</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 compartilhamento de pesos</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o kernel e os canais de uma camada CONV e compare com uma camada FC equivalente, para uma entrada hipotética de 32×32 pixels.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">kernel (k×k)</label><span id="ep0903_k_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">3</span></div>
        <input id="ep0903_k" style="width:100%;accent-color:#2980b9;" max="7" min="1" step="2" type="range" value="3">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">canais de entrada</label><span id="ep0903_cin_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">1</span></div>
        <input id="ep0903_cin" style="width:100%;accent-color:#2980b9;" max="16" min="1" step="1" type="range" value="1">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">canais de saída (filtros)</label><span id="ep0903_cout_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">8</span></div>
        <input id="ep0903_cout" style="width:100%;accent-color:#2980b9;" max="32" min="1" step="1" type="range" value="8">
      </div>
      <div style="display:flex;align-items:center;gap:8px;">
        <input id="ep0903_bias" type="checkbox" checked style="accent-color:#2980b9;width:16px;height:16px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">usar viés</label>
      </div>
    </div>
    <div id="ep0903_debug" style="background:#e3f2fd;border-radius:8px;padding:14px;border:1px solid #bbdefb;font-family:monospace;font-size:12px;color:#1565c0;text-align:center;line-height:1.8;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var kEl = root.querySelector('#ep0903_k'), kvEl = root.querySelector('#ep0903_k_v');
    var cinEl = root.querySelector('#ep0903_cin'), cinvEl = root.querySelector('#ep0903_cin_v');
    var coutEl = root.querySelector('#ep0903_cout'), coutvEl = root.querySelector('#ep0903_cout_v');
    var biasEl = root.querySelector('#ep0903_bias');
    var dbg = root.querySelector('#ep0903_debug');
    var LADO_ENTRADA = 32;

    function render(){
      var k = parseInt(kEl.value), cin = parseInt(cinEl.value), cout = parseInt(coutEl.value);
      var bias = biasEl.checked ? 1 : 0;
      kvEl.textContent = k+'×'+k; cinvEl.textContent = cin; coutvEl.textContent = cout;
      var paramsConv = k*k*cin*cout + cout*bias;
      var paramsFC = (LADO_ENTRADA*LADO_ENTRADA*cin)*cout + cout*bias;
      var razao = (paramsFC/paramsConv).toFixed(1);
      dbg.innerHTML =
        'CONV: '+k+'×'+k+'×'+cin+'×'+cout+' + '+cout+'×'+bias+'  =  <b>'+paramsConv.toLocaleString('pt-BR')+' parâmetros</b><br>' +
        'FC equivalente (entrada '+LADO_ENTRADA+'×'+LADO_ENTRADA+'×'+cin+' → '+cout+' saídas):  <b>'+paramsFC.toLocaleString('pt-BR')+' parâmetros</b><br>' +
        '↳ a camada FC teria <b>'+razao+'×</b> mais parâmetros que a CONV equivalente, apesar de "ver" a mesma entrada.';
    }
    [kEl,cinEl,coutEl,biasEl].forEach(function(el){ el.addEventListener('input', render); });
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0903');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.3:** Simulador: Contagem de Parâmetros — Convolução vs. Camada Totalmente Conectada


In [9]:
%%writefile EP09_03.py
# Código Python


Writing EP09_03.py


In [10]:
TestSuite("EP09_03.py").run()


### EP09_04 🟡 Interseção sobre União (IoU) e Supressão de Não-Máximos (NMS)

Tanto o Faster R-CNN quanto o YOLO, apresentados na seção "Aplicações em Larga Escala" e no Projeto Prático 2 deste capítulo, produzem internamente **muito mais** caixas delimitadoras candidatas do que objetos realmente existem na imagem — várias delas cobrindo, com pequenas variações, o mesmo objeto. A etapa de pós-processamento responsável por eliminar essas redundâncias é a **Supressão de Não-Máximos (NMS)**, e sua peça fundamental é a métrica de **Interseção sobre União (IoU)**, que mede o quanto duas caixas se sobrepõem — a mesma métrica usada, por exemplo, para definir o limiar de acerto do `mAP50` calculado no Projeto Prático 2 (IoU $\ge 0{,}5$ com a caixa verdadeira).

Você foi encarregado de implementar o algoritmo de NMS do zero: dado um conjunto de caixas candidatas com suas pontuações de confiança, filtrar as redundantes e manter apenas as detecções mais confiáveis e suficientemente distintas entre si.

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler o inteiro $N$ (número de caixas candidatas) e o limiar real $\tau$ (limiar de IoU para supressão).
2. **Caixas:** Ler $N$ linhas, cada uma com cinco valores reais: $x_1, y_1, x_2, y_2, \text{score}$ (canto superior-esquerdo, canto inferior-direito, pontuação de confiança).
3. **Interseção sobre União:** Para duas caixas $A$ e $B$,
$$
\text{IoU}(A,B) = \frac{\text{área}(A \cap B)}{\text{área}(A) + \text{área}(B) - \text{área}(A \cap B)},
$$
   onde a área de interseção é calculada pela sobreposição dos intervalos em $x$ e em $y$ (zero se não houver sobreposição).
4. **Algoritmo guloso de NMS:**
   a. Ordene as caixas por `score` decrescente (em caso de empate, mantenha a ordem de leitura original — ordenação estável).
   b. Selecione a caixa de maior pontuação entre as restantes; adicione-a ao conjunto de saída e remova-a da lista.
   c. Calcule o IoU dessa caixa selecionada com **todas** as caixas ainda restantes; remova (suprima) qualquer caixa com $\text{IoU} > \tau$.
   d. Repita os passos (b)–(c) até que não restem caixas.
5. **Saída:** Para cada caixa mantida, na ordem em que foi selecionada, imprimir seu índice original (posição de leitura, começando em $0$) e seu `score`, formatado com 4 casas decimais. Ao final, imprimir `Total mantidas: X`.

#### 📌 Restrições Computacionais

* **Supressão estrita:** apenas caixas com $\text{IoU} > \tau$ são suprimidas; caixas com $\text{IoU}$ exatamente igual a $\tau$ são **mantidas**.
* **Índices originais:** a saída referencia a posição em que cada caixa foi lida na entrada (começando em $0$), não sua posição após a ordenação.
* **Retângulos alinhados aos eixos:** todas as caixas são especificadas por dois cantos, com $x_1 < x_2$ e $y_1 < y_2$ garantidos na entrada.

#### 🧠 Fundamentação Teórica

| Elemento | Papel no pós-processamento de detecção |
|---|---|
| IoU | Quantifica a sobreposição espacial entre duas caixas; $\text{IoU}=1$ para caixas idênticas, $\text{IoU}=0$ para caixas disjuntas |
| Limiar $\tau$ | Controla a agressividade da supressão: $\tau$ baixo elimina até detecções pouco sobrepostas; $\tau$ alto preserva quase todas |
| Ordenação por confiança | Garante que, entre caixas redundantes, sempre sobrevive a de maior pontuação |
| `mAP50` (Projeto Prático 2) | Usa exatamente o mesmo limiar de IoU ($0{,}5$) para decidir se uma detecção final "acerta" a caixa verdadeira |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $N$ e real $\tau$.
* Próximas $N$ linhas: cinco reais $x_1\ y_1\ x_2\ y_2\ \text{score}$.

**Saída:**

* Uma linha por caixa mantida, na ordem de seleção: `índice score` (score com 4 casas decimais).
* Última linha: `Total mantidas: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3 0.5<br>0 0 10 10 0.9<br>1 1 11 11 0.75<br>50 50 60 60 0.8 | 0 0.9000<br>2 0.8000<br>Total mantidas: 2 | A caixa 1 é suprimida por sobrepor fortemente a caixa 0 (IoU≈0,68 > 0,5). |
| 2 0.5<br>0 0 10 10 0.9<br>0 0 10 6 0.95 | 1 0.9500<br>Total mantidas: 1 | A caixa 0 é suprimida por sobrepor demais a vencedora (IoU=0,6 > 0,5), mesmo tendo menor score de origem que a caixa 1, aqui a de maior score. |


In [11]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0904" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: IoU e Supressão de Não-Máximos</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 duas caixas candidatas</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste a sobreposição entre a caixa vencedora (azul, score maior) e a candidata (vermelha) e o limiar de supressão.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">deslocamento da candidata</label><span id="ep0904_dx_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">3</span></div>
        <input id="ep0904_dx" style="width:100%;accent-color:#2980b9;" max="10" min="0" step="1" type="range" value="3">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">limiar τ</label><span id="ep0904_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">0.50</span></div>
        <input id="ep0904_tau" style="width:100%;accent-color:#2980b9;" max="0.9" min="0.1" step="0.05" type="range" value="0.5">
      </div>
    </div>
    <div style="position:relative;width:100%;height:160px;background:#fafafa;border:1px solid #ddd;border-radius:12px;margin-bottom:16px;">
      <div id="ep0904_boxA" style="position:absolute;border:2px solid #2980b9;background:rgba(41,128,185,0.25);border-radius:4px;"></div>
      <div id="ep0904_boxB" style="position:absolute;border:2px solid #e74c3c;background:rgba(231,76,60,0.25);border-radius:4px;"></div>
    </div>
    <div id="ep0904_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var dxEl = root.querySelector('#ep0904_dx'), dxvEl = root.querySelector('#ep0904_dx_v');
    var tauEl = root.querySelector('#ep0904_tau'), tauvEl = root.querySelector('#ep0904_tau_v');
    var boxA = root.querySelector('#ep0904_boxA'), boxB = root.querySelector('#ep0904_boxB');
    var dbg = root.querySelector('#ep0904_debug');
    var ESCALA = 10; // px por unidade
    var A = {x1:5, y1:3, x2:15, y2:13, score:0.9};

    function iou(a,b){
      var ix1=Math.max(a.x1,b.x1), iy1=Math.max(a.y1,b.y1);
      var ix2=Math.min(a.x2,b.x2), iy2=Math.min(a.y2,b.y2);
      var iw=Math.max(0, ix2-ix1), ih=Math.max(0, iy2-iy1);
      var inter = iw*ih;
      var areaA=(a.x2-a.x1)*(a.y2-a.y1), areaB=(b.x2-b.x1)*(b.y2-b.y1);
      return inter/(areaA+areaB-inter);
    }

    function render(){
      var dx = parseInt(dxEl.value);
      var tau = parseFloat(tauEl.value);
      dxvEl.textContent = dx; tauvEl.textContent = tau.toFixed(2);
      var B = {x1:5+dx, y1:3+dx*0.4, x2:15+dx, y2:13+dx*0.4, score:0.75};

      boxA.style.left = (A.x1*ESCALA)+'px'; boxA.style.top = (A.y1*ESCALA)+'px';
      boxA.style.width = ((A.x2-A.x1)*ESCALA)+'px'; boxA.style.height = ((A.y2-A.y1)*ESCALA)+'px';
      boxB.style.left = (B.x1*ESCALA)+'px'; boxB.style.top = (B.y1*ESCALA)+'px';
      boxB.style.width = ((B.x2-B.x1)*ESCALA)+'px'; boxB.style.height = ((B.y2-B.y1)*ESCALA)+'px';

      var val = iou(A,B);
      var suprimida = val > tau;
      dbg.textContent = 'IoU(A,B) = '+val.toFixed(4)+'  |  τ = '+tau.toFixed(2)+'  →  candidata (vermelha) ' + (suprimida ? 'SUPRIMIDA' : 'MANTIDA');
    }
    dxEl.addEventListener('input', render);
    tauEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0904');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.4:** Simulador: IoU e Supressão de Não-Máximos


In [12]:
%%writefile EP09_04.py
# Código Python


Writing EP09_04.py


In [13]:
TestSuite("EP09_04.py").run()


### EP09_05 🔴 *Pipeline* Integrado: Da Detecção à Medição do Mundo Real

Este exercício final integra os dois exercícios anteriores ao princípio de **fotogrametria** apresentado na seção "Integrando Geometria e Aprendizado Profundo" — exatamente o mesmo cálculo implementado na figura de medição por referência de escala deste capítulo. O cenário reproduz uma situação realista: um detector (Faster R-CNN ou YOLO) gera **várias caixas candidatas sobrepostas** para o mesmo objeto de interesse; após filtrá-las por NMS, a caixa sobrevivente de maior confiança é usada, junto de uma caixa de referência de largura real conhecida (como o cartão de $8{,}56$ cm), para estimar as dimensões reais do objeto detectado.

#### 📋 Diretrizes de Implementação

1. **Referência conhecida:** Ler o valor real $L_{ref}$ (largura real do objeto de referência, em cm) e, em seguida, os quatro reais $x_1\ y_1\ x_2\ y_2$ de sua caixa delimitadora em pixels (já conhecida, sem necessidade de detecção).
2. **Candidatas do objeto a medir:** Ler o inteiro $N$ (número de caixas candidatas produzidas pelo detector para o objeto de interesse) e o limiar real $\tau$; em seguida, ler as $N$ linhas de caixas candidatas, cada uma com $x_1\ y_1\ x_2\ y_2\ \text{score}$.
3. **Etapa 1 — NMS:** Aplique exatamente o algoritmo de Supressão de Não-Máximos do EP09_04 às $N$ caixas candidatas, usando o limiar $\tau$, para eliminar detecções redundantes do mesmo objeto.
4. **Etapa 2 — Seleção da caixa final:** Após o NMS, a caixa de maior `score` entre as mantidas é a detecção final do objeto (a entrada garante que todas as caixas candidatas correspondem a um único objeto físico, portanto a primeira caixa selecionada pelo NMS já é o resultado final).
5. **Etapa 3 — Medição por referência de escala:** Calcule a razão $\text{cm/pixel} = L_{ref} / \text{largura da referência em pixels}$ e aplique-a tanto à largura quanto à altura (em pixels) da caixa final do objeto, obtendo suas dimensões reais estimadas em centímetros.
6. **Saída:** Primeiro, uma linha por caixa mantida após o NMS (mesmo formato do EP09_04): `índice score`. Em seguida, a linha `Total mantidas: X`. Por fim, a linha `Objeto: L x A cm`, onde $L$ e $A$ são a largura e a altura estimadas do objeto, cada uma com 2 casas decimais.

#### 📌 Restrições Computacionais

* **Reaproveite o NMS do EP09_04** integralmente — mesma regra de desempate, mesmo critério de supressão ($\text{IoU} > \tau$).
* **A referência não passa por NMS:** sua caixa é dada diretamente, sem candidatas concorrentes.
* **Razão única para largura e altura:** assim como na figura de fotogrametria do capítulo, a mesma razão cm/pixel (derivada da largura da referência) é aplicada tanto à largura quanto à altura do objeto — não há calibração vertical separada.

#### 🧠 Fundamentação Teórica

| Etapa | Conceito do capítulo |
|---|---|
| Múltiplas caixas candidatas | Saída bruta de um detector como o Faster R-CNN ou o YOLO, antes do pós-processamento |
| NMS (EP09_04) | Filtra a redundância, mantendo apenas a detecção mais confiável do objeto |
| Referência de escala conhecida | Mesmo princípio da fotogrametria: um objeto de dimensão real conhecida converte pixels em centímetros |
| Razão cm/pixel | Fator de conversão único, assumindo que a câmera está aproximadamente perpendicular à cena (sem correção de perspectiva) |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Real $L_{ref}$.
* Linha 2: Quatro reais $x_1\ y_1\ x_2\ y_2$ (caixa de referência).
* Linha 3: Inteiro $N$ e real $\tau$.
* Próximas $N$ linhas: cinco reais $x_1\ y_1\ x_2\ y_2\ \text{score}$ (caixas candidatas do objeto).

**Saída:**

* Uma linha por caixa mantida após NMS: `índice score` (score com 4 casas decimais).
* Linha `Total mantidas: X`.
* Linha final: `Objeto: L x A cm` (2 casas decimais cada).

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 8.56<br>30 200 170 288<br>3 0.5<br>250 100 470 250 0.92<br>255 105 468 245 0.88<br>600 600 650 650 0.40 | 0 0.9200<br>2 0.4000<br>Total mantidas: 2<br>Objeto: 13.45 x 9.17 cm | A caixa 1 é suprimida por sobrepor fortemente a caixa 0; a detecção final do objeto é a caixa 0. |


In [14]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0905" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Pipeline Integrado</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 detecção + fotogrametria</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">A caixa cinza é a referência de largura real conhecida; ajuste-a e veja a medição do objeto (vermelho) recalculada — a mesma razão cm/pixel se aplica à largura e à altura.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Largura real da referência (cm)</label>
        <span id="ep0905_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">8.56</span>
      </div>
      <input id="ep0905_sl" style="width:100%;accent-color:#2980b9;" max="15" min="3" step="0.1" type="range" value="8.56">
    </div>
    <div style="position:relative;width:100%;height:180px;background:#fafafa;border:1px solid #ddd;border-radius:12px;margin-bottom:16px;">
      <div id="ep0905_ref" style="position:absolute;left:20px;top:110px;width:140px;height:88px;border:2px solid #7f8c8d;background:rgba(200,200,200,0.6);border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:10px;color:#444;font-family:monospace;">referência</div>
      <div id="ep0905_obj" style="position:absolute;left:220px;top:30px;width:180px;height:120px;border:2px solid #e74c3c;background:rgba(231,76,60,0.25);border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:10px;color:#a12;font-family:monospace;">objeto detectado</div>
    </div>
    <div id="ep0905_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var slEl = root.querySelector('#ep0905_sl');
    var vlEl = root.querySelector('#ep0905_vl');
    var dbg = root.querySelector('#ep0905_debug');
    var W_REF_PX = 140, H_REF_PX = 88;
    var W_OBJ_PX = 180, H_OBJ_PX = 120;

    function render(){
      var lref = parseFloat(slEl.value);
      vlEl.textContent = lref.toFixed(2);
      var razao = lref / W_REF_PX;
      var largObj = W_OBJ_PX * razao;
      var altObj = H_OBJ_PX * razao;
      dbg.textContent = 'razão = '+lref.toFixed(2)+' / '+W_REF_PX+'px = '+razao.toFixed(4)+' cm/px  →  objeto ≈ '+largObj.toFixed(2)+' x '+altObj.toFixed(2)+' cm';
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0905');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.5:** Simulador: Pipeline Integrado — NMS + Medição por Referência de Escala


In [15]:
%%writefile EP09_05.py
# Código Python


Writing EP09_05.py


In [16]:
TestSuite("EP09_05.py").run()
